# TerraAlert — Wildfire Dataset EDA
**Dataset 1:** NASA FIRMS (Fire Information for Resource Management System)  
**Dataset 2:** 1.88 Million US Wildfires (Kaggle)  
**Research Question:** Do gradient boosting methods (XGBoost/LightGBM) outperform Random Forest for tabular wildfire risk prediction?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1A. NASA FIRMS Data (Satellite Hotspots)
Get your free API key at: https://firms.modaps.eosdis.nasa.gov/api/  
Takes 2 minutes — just enter your email.

In [ ]:
# Replace with your free NASA FIRMS API key
NASA_FIRMS_API_KEY = 'YOUR_KEY_HERE'

# FIRMS API - last 10 days of fire hotspots for USA
# Format: area/csv/{api_key}/{source}/{area}/{days}
url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{NASA_FIRMS_API_KEY}/VIIRS_SNPP_NRT/-130,24,-65,50/10'

print('Downloading NASA FIRMS fire hotspot data...')
response = requests.get(url)

if response.status_code == 200:
    with open('../data/wildfires/nasa_firms_hotspots.csv', 'wb') as f:
        f.write(response.content)
    print('Download complete ✅')
else:
    print(f'Error {response.status_code} — check your API key')
    print('Get free key at: https://firms.modaps.eosdis.nasa.gov/api/')

## 1B. Kaggle Wildfire Dataset
Download manually from: https://www.kaggle.com/datasets/rtatman/188-million-us-wildfires  
Save the CSV to: `../data/wildfires/us_wildfires.csv`  
Free Kaggle account needed.

In [ ]:
# Load NASA FIRMS data
try:
    firms_df = pd.read_csv('../data/wildfires/nasa_firms_hotspots.csv')
    print(f'NASA FIRMS shape: {firms_df.shape}')
    print(f'Columns: {list(firms_df.columns)}')
    firms_df.head()
except:
    print('NASA FIRMS file not found — run the download cell above first')

In [ ]:
# Load Kaggle wildfire data
try:
    wf_df = pd.read_csv('../data/wildfires/us_wildfires.csv')
    print(f'Kaggle Wildfire shape: {wf_df.shape}')
    print(f'Columns: {list(wf_df.columns)}')
    wf_df.head()
except:
    print('Kaggle file not found — download from link above')

## 2. Data Quality Check

In [ ]:
# Run whichever dataset you have loaded
df = wf_df  # or firms_df

print('=== DATASET OVERVIEW ===')
print(f'Total records: {len(df):,}')
print(f'\nNull values (top 10):')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print(f'\nData types:')
print(df.dtypes.value_counts())

## 3. Exploratory Analysis

In [ ]:
# Kaggle dataset EDA (adjust column names if using FIRMS)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('TerraAlert — Wildfire Dataset EDA', fontsize=16, fontweight='bold')

# Plot 1: Fire size distribution
if 'FIRE_SIZE' in df.columns:
    fire_size = df['FIRE_SIZE'].dropna()
    axes[0,0].hist(np.log1p(fire_size), bins=40, color='#e67e22', edgecolor='white', alpha=0.8)
    axes[0,0].set_title('Fire Size Distribution (log scale)')
    axes[0,0].set_xlabel('log(Fire Size in Acres)')
    axes[0,0].set_ylabel('Count')

# Plot 2: Fires per year
if 'FIRE_YEAR' in df.columns:
    yearly = df.groupby('FIRE_YEAR').size()
    axes[0,1].bar(yearly.index, yearly.values, color='#e74c3c', edgecolor='white')
    axes[0,1].set_title('Wildfires Per Year')
    axes[0,1].set_xlabel('Year')
    axes[0,1].set_ylabel('Count')

# Plot 3: Fire cause
if 'STAT_CAUSE_DESCR' in df.columns:
    cause_counts = df['STAT_CAUSE_DESCR'].value_counts().head(8)
    axes[1,0].barh(cause_counts.index, cause_counts.values, color='#f39c12')
    axes[1,0].set_title('Top 8 Fire Causes')
    axes[1,0].set_xlabel('Count')

# Plot 4: Fire class distribution
if 'FIRE_SIZE_CLASS' in df.columns:
    class_counts = df['FIRE_SIZE_CLASS'].value_counts()
    axes[1,1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
                  colors=plt.cm.Oranges(np.linspace(0.3, 0.9, len(class_counts))))
    axes[1,1].set_title('Fire Size Class Distribution')

plt.tight_layout()
plt.savefig('../data/wildfires/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved ✅')

## 4. Model Justification

**For this wildfire dataset, I will compare:**

| Model | Reason |
|---|---|
| **Random Forest** | Baseline — well-established, handles mixed feature types |
| **XGBoost** | Standard upgrade for tabular data, handles missing values natively |
| **LightGBM** | Faster than XGBoost on large datasets, leaf-wise tree growth |

**Target variable:** Fire size class (A–G) — multi-class classification  
**Features:** Location, month, cause, vegetation type, weather conditions  
**Evaluation metrics:** F1-score (weighted), Precision, Recall, AUC-ROC  
**Expected outcome:** LightGBM fastest, XGBoost highest accuracy, RF as interpretable baseline


In [ ]:
# Feature engineering preview
if 'FIRE_YEAR' in df.columns:
    feature_cols = ['LATITUDE', 'LONGITUDE', 'FIRE_YEAR', 'STAT_CAUSE_CODE']
    if 'DISCOVERY_DOY' in df.columns:
        feature_cols.append('DISCOVERY_DOY')
    
    available = [c for c in feature_cols if c in df.columns]
    df_model_ready = df[available + ['FIRE_SIZE_CLASS']].dropna()
    
    print(f'Model-ready dataset shape: {df_model_ready.shape}')
    print(f'Target class distribution:')
    print(df_model_ready['FIRE_SIZE_CLASS'].value_counts())
    
    df_model_ready.to_csv('../data/wildfires/wildfires_processed.csv', index=False)
    print('\nProcessed dataset saved ✅')